In [ ]:
# --- paths come from human/config.py (auto-inserted by fix_notebooks.py) ---
import sys; sys.path.append('..')
from config import HUMAN_BASE


# Build Promoter Strength File (FANTOM5 CAGE-seq)

**Goal**: Produce `BMMC_promoter_strength.txt` — BMMC equivalent of yeast pipeline's `GRN_ssTFs_Sc_promoter_strength.txt`.

**Why FANTOM5 CAGE (not GTEx mRNA-seq)**:
Yeast pipeline uses Pelechano 2013 CAGE-seq YPD condition TPM, which is **nascent transcription** based on 5'-cap trapping. GTEx is polyA mRNA-seq which conflates transcription rate with mRNA stability — wrong proxy for promoter activity. FANTOM5 uses the same CAGE technology as Pelechano yeast data and covers hematopoietic primary cells — the direct human analog.

**Approach**:
1. Download FANTOM5 hg38 reprocessed v8 CAGE peak expression matrix
2. Download CAGE peak → gene symbol annotation
3. Map our 88 genes to their CAGE peaks (a gene may have multiple peaks)
4. Identify FANTOM5 hematopoietic primary cell samples (CD34+, CD14+, CD19+, CD8+, CD4+, NK, etc.)
5. Aggregate: per gene per sample TPM = sum of all CAGE peaks for that gene
6. Take max across hematopoietic samples (= the strongest condition this promoter can drive) = single per-gene CAGE-TPM scalar (= maximal nascent promoter activity observed in hematopoietic lineages)
7. Write in yeast format: one row per BMMC cell type, all rows identical (= per-gene max), tab-separated

## Part A: Setup, download FANTOM5 data

In [ ]:
import gzip
import urllib.request
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
import re

GENE_LIST_PATH = f"{HUMAN_BASE}/bmmc_gene_list.tsv"
EXPR_MATRIX_PATH = f"{HUMAN_BASE}/GTEx_v11/setia_input_linear_CPM.tsv"   # for getting BMMC cell type names (Samples_Dic keys)

OUTPUT_DIR  = Path(f"{HUMAN_BASE}/GTEx_v11/promoter_strength_output");  OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR   = Path(f"{HUMAN_BASE}/GTEx_v11/promoter_strength_cache");   CACHE_DIR.mkdir(exist_ok=True)

# FANTOM5 hg38 v8 files
FANTOM5_BASE = 'https://fantom.gsc.riken.jp/5/datafiles/reprocessed/hg38_v8/extra'
TPM_URL    = f'{FANTOM5_BASE}/CAGE_peaks_expression/hg38_fair+new_CAGE_peaks_phase1and2_tpm_ann.osc.txt.gz'
ANNOT_URL  = f'{FANTOM5_BASE}/CAGE_peaks_annotation/hg38_liftover+new_CAGE_peaks_phase1and2_annot.txt.gz'

TPM_PATH   = CACHE_DIR / 'fantom5_hg38_tpm_ann.osc.txt.gz'
ANNOT_PATH = CACHE_DIR / 'fantom5_hg38_peaks_annot.txt.gz'

# Load BMMC gene list (88 genes in correct order)
gene_df = pd.read_csv(GENE_LIST_PATH, sep='\t', comment='#')
all_genes = gene_df['gene_symbol'].tolist()
print(f'BMMC panel: {len(all_genes)} genes')
print(f'First 5: {all_genes[:5]}')
print(f'Last 5:  {all_genes[-5:]}')

# Load BMMC cell type order (26 cell types) — needed for output rows
expr = pd.read_csv(EXPR_MATRIX_PATH, sep='\t', index_col=0)
bmmc_cell_types = expr.index.tolist()
print(f'\nBMMC cell types ({len(bmmc_cell_types)}):')
for ct in bmmc_cell_types:
    print(f'  {ct}')

In [ ]:
# Download FANTOM5 files (cached locally, ~few hundred MB total)
for url, path in [(ANNOT_URL, ANNOT_PATH), (TPM_URL, TPM_PATH)]:
    if not path.exists():
        print(f'Downloading {url.split("/")[-1]}...')
        urllib.request.urlretrieve(url, path)
    sz_mb = path.stat().st_size / 1e6
    print(f'  {path.name}: {sz_mb:.1f} MB')

## Part B: Parse peak → gene annotation

In [ ]:
# === Gene aliases + coordinate-based rescue for FANTOM5 annotation gaps ===
# FANTOM5 hg38 v8 reprocessed annotation lost gene assignment in some highly
# repetitive regions (e.g. alpha-globin cluster on chr16p13.3). Peak IDs in
# the file are still hg19 coordinates (format: 'hg19::chr16:227386..227428,+;hg_xxx').
# We resolve missing gene assignment by direct hg19 coordinate matching for
# known panel genes whose CAGE peaks are unannotated in FANTOM5 hg38 v8.

GENE_ALIASES = {
    'HBA2': 'HBA1',
    'HBM' : 'HBA1',   # but careful: LRP5 also has HBM synonym, won't match this way
}

# Coordinate-based rescue (hg19 coordinates):
# Any peak with empty annotation falling in these intervals is assigned to the listed gene.
# Alpha-globin TSS positions per UCSC/RefSeq hg19:
#   HBA1: chr16:226,679-227,521 (forward strand)
#   HBA2: chr16:222,846-223,710 (forward strand)
# Note: HBA1 and HBA2 are paralogs ~3 kb apart with near-identical sequence.
# Our panel contains HBA1; we include the broader region covering both since
# they share promoter elements and reads cannot be distinguished by alignment alone.
COORD_RESCUE = [
    # (chrom, start, end, gene_to_assign)
    ('chr16', 222000, 228500, 'HBA1'),   # HBA2 + HBA1 promoter region (hg19)
]

PEAK_COORD_RE = re.compile(r'hg(?:19|38)::([^:]+):(\d+)\.\.(\d+),')

def parse_peak_coords(peak_id):
    """Extract (chrom, start, end) from FANTOM5 peak_id, or None."""
    m = PEAK_COORD_RE.match(peak_id)
    if not m:
        return None
    return (m.group(1), int(m.group(2)), int(m.group(3)))

def rescue_by_coord(peak_id):
    """If peak falls in a known rescue interval, return the gene; else None."""
    coords = parse_peak_coords(peak_id)
    if coords is None:
        return None
    chrom, start, end = coords
    # Use peak midpoint for assignment
    mid = (start + end) // 2
    for r_chrom, r_start, r_end, r_gene in COORD_RESCUE:
        if chrom == r_chrom and r_start <= mid <= r_end:
            return r_gene
    return None

# === Build peak_to_gene with all fallbacks ===
gene_targets = set(all_genes)
peak_to_gene = {}
all_annotated_symbols = set()
rescue_count = 0
alias_count = 0
synonym_count = 0

print('Parsing peak annotation (with alias and coordinate rescue)...')
with gzip.open(ANNOT_PATH, 'rt', errors='replace') as f:
    header = None
    for line in f:
        if line.startswith('#') or line.startswith('00Annotation'):
            continue
        parts = line.rstrip('\n').split('\t')
        if header is None:
            header = parts
            print(f'  Header: {header[:10]}')
            continue
        if len(parts) < 8:
            continue
        peak_id       = parts[0]
        gene_symbol   = parts[7] if len(parts) > 7 else ''
        gene_synonyms = parts[8] if len(parts) > 8 else ''
        all_annotated_symbols.add(gene_symbol)

        # 1. Direct symbol match
        if gene_symbol in gene_targets:
            peak_to_gene[peak_id] = gene_symbol
            continue
        # 2. Alias
        if gene_symbol in GENE_ALIASES:
            resolved = GENE_ALIASES[gene_symbol]
            if resolved in gene_targets:
                peak_to_gene[peak_id] = resolved
                alias_count += 1
                continue
        # 3. Synonym (space-separated in FANTOM5)
        if gene_synonyms:
            for syn in re.split(r'[\s,]+', gene_synonyms):
                if syn in gene_targets:
                    peak_to_gene[peak_id] = syn
                    synonym_count += 1
                    break
            if peak_id in peak_to_gene:
                continue
        # 4. Coordinate-based rescue (only if symbol/synonym empty)
        if gene_symbol == '':
            rescued = rescue_by_coord(peak_id)
            if rescued is not None and rescued in gene_targets:
                peak_to_gene[peak_id] = rescued
                rescue_count += 1

print(f'\nMapping summary:')
print(f'  Total peaks mapped to panel: {len(peak_to_gene)}')
print(f'    Direct symbol match: {len(peak_to_gene) - alias_count - synonym_count - rescue_count}')
print(f'    Alias-resolved:      {alias_count}')
print(f'    Synonym-resolved:    {synonym_count}')
print(f'    Coord-rescued:       {rescue_count}')

from collections import Counter
gene_peak_counts = Counter(peak_to_gene.values())
print(f'\nGenes with at least one CAGE peak: {len(gene_peak_counts)} / {len(all_genes)}')
missing = [g for g in all_genes if g not in gene_peak_counts]
if missing:
    print(f'MISSING: {missing}')

# Show HBA1 specifically
if 'HBA1' in gene_peak_counts:
    print(f'\nHBA1: {gene_peak_counts["HBA1"]} peaks (via coord rescue or direct)')
    hba1_peaks = [pid for pid, g in peak_to_gene.items() if g == 'HBA1']
    for p in hba1_peaks[:10]:
        print(f'  {p[:100]}')

print(f'\nPeaks per gene (top 10):')
for g, c in gene_peak_counts.most_common(10):
    print(f'  {g}: {c} peaks')

## Part C: Parse TPM matrix — stream over header to find hematopoietic samples

In [ ]:
# TPM file is huge (~few GB uncompressed). Stream parse.
# First pass: just read header to identify hematopoietic primary cell columns.
# Sample column names format: 'tpm.<sample_description>.CNhsXXXXX.<library_id>.<assembly>...'

# Hematopoietic primary cell keywords from FANTOM5 sample collection.
# These keywords appear in the column headers (after URL-decoded percent encoding).
HEMATO_KEYWORDS = [
    'CD34',                  # HSPC
    'CD14',                  # Monocyte
    'CD19',                  # B cell
    'CD8',                   # CD8 T cell
    'CD4',                   # CD4 T cell
    'natural killer',        # NK
    'CD56',                  # NK / NKT alt name
    'neutrophil',
    'eosinophil',
    'basophil',
    'macrophage',
    'dendritic cell',        # DC
    'plasmacytoid',          # pDC
    'monocyte',              # broader monocyte
    'B cell',                # broader B
    'T cell',                # broader T
    'erythrocyte',
    'erythroblast',
    'mast cell',
    'megakaryocyte',
    'lymphocyte',
    'granulocyte',
    'myeloid',
]
# Filter OUT (cell lines and disease samples bias the baseline)
EXCLUDE_KEYWORDS = [
    'cell line',
    'leukemia',
    'lymphoma',
    'myeloma',
    'tumor',
    'cancer',
    'differentiation',     # time course; not baseline
    'infection',
    'stimulated',
    'treated',
]

import urllib.parse

def is_hemato(col_name):
    if not col_name.startswith('tpm.'):
        return False
    desc = urllib.parse.unquote(col_name[4:].split('.CNhs')[0]).lower()
    for ex in EXCLUDE_KEYWORDS:
        if ex.lower() in desc:
            return False
    for kw in HEMATO_KEYWORDS:
        if kw.lower() in desc:
            return True
    return False

# Stream the TPM file: find header line, then identify hematopoietic columns
header_cols = None
with gzip.open(TPM_PATH, 'rt', errors='replace') as f:
    for line in f:
        if line.startswith('##'):
            continue
        # First non-meta line is the data header (starts with '00Annotation' or column names)
        if line.startswith('00Annotation') or line.startswith('eedb:annotation'):
            header_cols = line.rstrip('\n').split('\t')
            break
        if line.startswith('tpm.') or '\ttpm.' in line:
            header_cols = line.rstrip('\n').split('\t')
            break

print(f'Total columns: {len(header_cols)}')
print(f'First 3 columns: {header_cols[:3]}')

# Identify hematopoietic sample column indices
hemato_col_idx = [i for i, c in enumerate(header_cols) if is_hemato(c)]
print(f'\nHematopoietic sample columns identified: {len(hemato_col_idx)}')
print(f'\nFirst 10 hematopoietic sample names (URL-decoded):')
for i in hemato_col_idx[:10]:
    desc = urllib.parse.unquote(header_cols[i][4:].split('.CNhs')[0])
    print(f'  [{i}] {desc}')

In [ ]:
# Second pass: stream all data rows, only keep rows where peak_id is in peak_to_gene.
# Per row, extract TPM values for hematopoietic columns.
# Per (peak, sample): TPM value. Per (gene, sample): sum of TPMs across all peaks of that gene.

peak_id_col = 0   # First column = peak ID

# gene -> list of (peak_id, np.array of TPMs across hemato samples)
gene_peak_tpms = defaultdict(list)
n_rows_total = 0
n_rows_kept = 0

print('Stream-parsing TPM matrix (this may take a few minutes)...')
import time
t0 = time.time()

with gzip.open(TPM_PATH, 'rt', errors='replace') as f:
    found_header = False
    for line in f:
        if line.startswith('##'):
            continue
        if not found_header:
            # Skip the header line (we already parsed it above)
            found_header = True
            continue
        n_rows_total += 1
        if n_rows_total % 50000 == 0:
            print(f'  Processed {n_rows_total:,} rows, {n_rows_kept:,} kept ({time.time()-t0:.0f}s)')
        # Quick peek: peak_id is first column
        tab_pos = line.find('\t')
        peak_id = line[:tab_pos]
        if peak_id not in peak_to_gene:
            continue
        # Parse full row
        parts = line.rstrip('\n').split('\t')
        try:
            tpms = np.array([float(parts[i]) if parts[i] else 0.0 for i in hemato_col_idx])
        except (ValueError, IndexError):
            continue
        gene = peak_to_gene[peak_id]
        gene_peak_tpms[gene].append((peak_id, tpms))
        n_rows_kept += 1

print(f'\nDone in {time.time()-t0:.0f}s')
print(f'Total rows: {n_rows_total:,}')
print(f'Rows kept (peak in panel): {n_rows_kept:,}')
print(f'Genes with at least one peak with TPM data: {len(gene_peak_tpms)}')

## Part D: Aggregate to per-gene CAGE-TPM scalar

In [ ]:
# For each gene, per sample: sum TPM across all CAGE peaks (mature gene-level CAGE TPM)
# Then per gene: MAX across hematopoietic samples (promoter strength = best-case drive)

gene_to_max_tpm = {}
for gene in all_genes:
    if gene not in gene_peak_tpms:
        gene_to_max_tpm[gene] = 0.0   # fallback if not in FANTOM5
        continue
    # Sum across peaks per sample, get array of shape (n_hemato_samples,)
    peak_arrays = [tpms for _, tpms in gene_peak_tpms[gene]]
    per_sample_total = np.sum(peak_arrays, axis=0)
    # MAX across samples — promoter strength is the peak driving capacity,
    # not the average. A lineage-restricted gene (e.g. HBB only in erythroid) has
    # low median but high max, which correctly reflects its strong promoter.
    gene_to_max_tpm[gene] = float(np.max(per_sample_total))

# Summary
vals = np.array([gene_to_max_tpm[g] for g in all_genes])
print(f'Gene-level MAX CAGE-TPM stats (= promoter strength proxy):')
print(f'  min:   {vals.min():.2f}')
print(f'  max:   {vals.max():.2f}')
print(f'  median: {np.median(vals):.2f}')
print(f'  mean:   {vals.mean():.2f}')
print(f'\nGenes with zero TPM: {(vals == 0).sum()} (likely no peak or low-expression in FANTOM5 hematopoietic samples)')

# Sanity check on key hematopoietic markers — should be high in their lineage cells
print(f'\nSanity check on selected genes:')
for g in ['HBB', 'HBA1', 'CD14', 'CD19', 'CD3D', 'CD4', 'CD8A', 'GATA1', 'SPI1', 'PAX5', 'RUNX1', 'NKG7', 'MPO']:
    if g in gene_to_max_tpm:
        print(f'  {g:6s}: max CAGE-TPM = {gene_to_max_tpm[g]:>8.2f}')

## Part E: Write yeast-format promoter strength file

In [ ]:
# Yeast format (from GRN_input_acquisition.py lines 1068-1077):
#   For each sample (cell type) in sorted(Samples_Dic.keys()):
#     For each gene in Column_order:
#       write str(promoter_strength)
#       tab between, no trailing tab
#     newline
# All rows have IDENTICAL content (single per-gene CAGE TPM value, not cell-type-specific).

out_path = OUTPUT_DIR / 'BMMC_promoter_strength.txt'
with open(out_path, 'w') as f:
    for cell_type in sorted(bmmc_cell_types):
        values = [str(gene_to_max_tpm[g]) for g in all_genes]
        f.write('\t'.join(values))
        f.write('\n')

n_bytes = out_path.stat().st_size
print(f'Saved: {out_path}  ({n_bytes:,} bytes)')

# Verify
with open(out_path) as f:
    lines = f.readlines()
print(f'\nLines: {len(lines)}  (expected {len(bmmc_cell_types)} = n cell types)')
tokens_per_line = [len(line.rstrip('\n').split('\t')) for line in lines]
print(f'Tokens per line: {set(tokens_per_line)}  (expected {{{len(all_genes)}}})')
all_same = len(set(lines)) == 1
print(f'All rows identical: {all_same}  (expected True — promoter strength is cell-type-independent)')

# Save companion lookup
lookup_path = OUTPUT_DIR / 'BMMC_promoter_strength_lookup.tsv'
with open(lookup_path, 'w') as f:
    f.write('order_idx\tgene_symbol\tcage_tpm_max_hemato\tn_cage_peaks\n')
    for i, g in enumerate(all_genes):
        n_peaks = len(gene_peak_tpms.get(g, []))
        f.write(f'{i}\t{g}\t{gene_to_max_tpm[g]:.4f}\t{n_peaks}\n')
print(f'Saved: {lookup_path}')

print('\nFirst 200 chars of output file:')
print(lines[0][:200])

In [ ]:
# Methods summary for paper
print('=' * 60)
print('Methods summary:')
print('=' * 60)
n_hemato = len(hemato_col_idx)
print(f'''
Promoter strength values were obtained from the FANTOM5 expression atlas
(hg38 v8 reprocessed release), which quantifies nascent transcription via
CAGE (Cap Analysis of Gene Expression) and is the direct human analog of
the yeast pipeline's Pelechano 2013 CAGE-seq YPD data. We used the
phase1+2 CAGE peak TPM matrix and gene-symbol annotation. For each of the
88 panel genes, all FANTOM5 CAGE peaks assigned to that gene were summed
to produce a gene-level CAGE-TPM in each sample. We then identified
{n_hemato} FANTOM5 samples corresponding to hematopoietic primary cells
(by keyword match on sample descriptions, excluding cell lines, disease
states, and perturbation samples), and took the MAX gene-level CAGE-
TPM across these samples as the per-gene promoter strength scalar.
The maximum (rather than mean or median) is used because promoter strength
represents the gene's peak nascent transcription capacity across the
relevant cell-type space, not its average expression level — a lineage-
restricted gene like HBB (high only in erythroid) has a weak average but
a strong promoter, correctly captured by the max.
Following yeast pipeline convention, promoter strength is treated as
cell-type-independent: the same per-gene vector is repeated for each
BMMC cell type row in the output file.
''')